# B2-023 — Generative Models and Diffusion

*30 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

This double-length unit teaches two families of generative models and the system that combines them with earlier Book 2 components: generative adversarial networks (GANs), denoising diffusion probabilistic models (DDPMs), and the Stable Diffusion pipeline.
Every model is a tiny network trained from scratch on CPU from fixed seeds; Stable Diffusion is taught as an architecture and practised through faithful CPU-scale analogues.

## Learning goals

- Define the generator and discriminator, write the minimax value function with BCE-with-logits, derive the optimal discriminator by pointwise optimization, relate the game's optimum to the Jensen–Shannon divergence, and explain the non-saturating generator loss.
- Train a GAN with alternating updates, separate optimizers, and `detach` discipline; read loss traces correctly; diagnose mode collapse and discriminator overpowering with a mode-coverage diagnostic.
- Build a variance schedule, derive the closed form $q(x_t\mid x_0)$ with the sum rule for independent Gaussians, and explain why $\bar\alpha_T\approx0$.
- Use the stated forward-process posterior to derive the ε-prediction objective for $t\ge2$, explain the $t=1$ surrogate, train a tiny DDPM, and sample with $\sigma_t^2=\tilde\beta_t$ and a noiseless final step.
- Explain latent diffusion, the latent scaling factor, cross-attention conditioning, conditioning dropout, and classifier-free guidance; map every Stable Diffusion component to its role; train a class-conditional latent diffusion analogue.

## Owned concepts

| Concept | First taught |
|---|---|
| `generative-adversarial-network` | Sessions 1–2 |
| `denoising-diffusion-probabilistic-models` | Sessions 3–4 |
| `stable-diffusion` | Session 5 |

## Study route

0. [Book 1 and Book 2 bridge diagnostic](./lessons/00-book1-bridge.ipynb) — 30 minutes.
1. [Adversarial games](./lessons/01-adversarial-games.ipynb) — 90 minutes. Then p01, p02, p06, p13.
2. [Training and diagnosing GANs](./lessons/02-training-and-diagnosing-gans.ipynb) — 90 minutes. Then p07, p08, p14, p17, p22.
3. [The diffusion forward process](./lessons/03-diffusion-forward-process.ipynb) — 90 minutes. Then p03, p09, p15.
4. [Denoising and sampling](./lessons/04-denoising-and-sampling.ipynb) — 90 minutes. Then p04, p10, p11, p16, p18, p19.
5. [Latent and conditional diffusion](./lessons/05-latent-and-conditional-diffusion.ipynb) — 90 minutes. Then p05, p12, p20, p21, p23, p24.
6. [Review and self-quiz](./review.ipynb) — 60 minutes.

Practice sets: Set A (p01–p05) is retrieval multiple choice; Set B (p06–p16) is constrained coding and proof; Set C (p17–p24) is integrative training, scenario analysis, and challenge repair/audit.

## Tools taught inside this unit

Book 1 and the earlier Book 2 units stop short of several tools this unit needs, so the lessons teach them before any practice uses them: the sigmoid and binary cross-entropy with logits as the two-class case of cross-entropy; KL divergence for densities and the Jensen–Shannon divergence; pointwise optimization of an integral objective under a positive-density assumption, with a boundary convention; the sum rule for independent Gaussians; the stated forward-process posterior $q(x_{t-1}\mid x_t,x_0)$ with the equal-variance KL argument for $t\ge2$ and the $t=1$ surrogate; the sampling variance $\sigma_t^2=\tilde\beta_t$; classifier-free guidance with conditioning dropout; and cross-attention conditioning.

## Reproducibility contract

The sole seed is `20261015`. The datasets `mixture2d` (four 2-D Gaussian blobs with known centres; 256 training and 64 held-out rows) and `latent4d` (three classes of 4-D rows generated as `decode(z)` from 2-D latents; 64 training and 16 held-out rows per class), the frozen orthonormal encoder matrix, the class centres, and `LATENT_SCALE = 0.25` are produced by `scripts/generate_generative_data.py` and stored in `data/generative_datasets.json`. Load them with `data/generative_data.py`, which verifies every row's SHA-256, checks that the frozen decoder reproduces every `latent4d` row exactly, and exposes immutable `TRAIN_IDS` and `HELDOUT_IDS` (row $i$ is held out exactly when $i \bmod 5=4$). Running `scripts/generate_generative_data.py --check` regenerates the rows and compares hashes. Training practices use CPU float32 tensors, full-batch Adam with the learning rate and step count stated in each statement, noise drawn from dedicated seeded generators and passed as explicit arguments, no shuffling or schedulers, and no stored trained weights, samples, or final metrics.

## How to study

Run lessons in order, answer both checkpoints in every numbered section before reading its collected answers, and attempt each practice only after the session listed in the study route. Proofs require reasoning in full sentences; coding statements pin function names, shapes, dtypes, seeds, and tolerances, and every coding response ends with an answer-check section of executable asserts.

## Boundaries

MLP generators, discriminators, and denoisers only; a single cross-attention block stands in for Stable Diffusion's U-Net cross-attention. No external data, torchvision datasets, model hubs, pretrained weights, GPU work, or downloads. Transposed-convolution (DCGAN) generators, Wasserstein GANs, score-based and continuous-time diffusion, fast samplers, and real Stable Diffusion inference appear only as forward-only enrichment; the GPU workflow belongs to `B2-024`.